# Overview

This notebook is a Kaggle-submit-ready **robust hybrid** fork of the 0.946 ProtoSSM + SED notebook.

It keeps the safer seed `42`, the multi-view post-blend, branch validation, and optional CV9245 sidecar, while upgrading BirdNET to the stronger 3-second chunk direct/proxy mapping branch. Rare-taxon suppression remains a saved ablation variant, not the primary submission.


# Required Kaggle Inputs

| Type | Slug / Source | Purpose | Required? |
|---|---|---|---|
| Competition | `birdclef-2026` | Test soundscapes, taxonomy, sample submission, labeled train soundscapes | Yes |
| Dataset | `tuckerarrants/bc2026-distilled-sed-public` | 5-fold public SED ONNX ensemble | Yes |
| Dataset | `jaejohn/perch-meta` | Cached Perch logits and embeddings for labeled train soundscapes | Yes |
| Dataset | `rishikeshjani/perch-onnx-for-birdclef-2026` | ONNX Runtime wheel and Perch ONNX fallback assets | Yes |
| Dataset | `tuckerarrants/perch-v2-no-dft-onnx` or `nina2025/perch-v2-no-dft-v3-onnx` | Preferred no-DFT Perch v2 ONNX backbone | Strongly recommended |
| Dataset | `chaneyma/birdclef-2026-cv9245-moe-artifacts` | Public CV9245 sidecar branch | Optional |
| Model | `shadiakiki1/birdnet-analyzer` | Stronger public BirdNET mapped branch | Strongly recommended |
| Model | `google/bird-vocalization-classifier` / TensorFlow2 `perch_v2_cpu` | Perch labels and TensorFlow fallback | Yes |
| Notebook/Dataset | `ashok205/tf-wheels` | TensorFlow 2.20 offline wheels for Kaggle CPU runtime | Yes |

Manual CV9245 input link:

https://www.kaggle.com/datasets/chaneyma/birdclef-2026-cv9245-moe-artifacts

Optional no-DFT input link:

https://www.kaggle.com/datasets/nina2025/perch-v2-no-dft-v3-onnx

BirdNET model link:

https://www.kaggle.com/models/shadiakiki1/birdnet-analyzer


# Architecture

```text
ONNX Perch v2 no-DFT
    ├─ competition-class logits
    └─ 1536-d embeddings

Labeled train soundscapes
    ├─ duplicate label cleanup
    ├─ site/hour priors
    ├─ PCA + MLP probes
    ├─ ProtoSSM sequence model
    └─ ResidualSSM correction

Hidden test soundscapes
    ├─ ProtoSSM/probe/residual branch → submission_protossm.csv
    ├─ Tucker public SED branch       → submission_sed.csv
    ├─ optional CV9245 public sidecar
    └─ optional BirdNET public mapped-Aves branch

Validated public branch registry
    ├─ row_id alignment checks
    ├─ finite [0, 1] checks
    ├─ mapped-class masks where needed
    └─ rank blend + sonotype mirroring → submission.csv
```

# Design Notes

- **Robust target:** preserve the cleaner 0.946 notebook structure while importing the strongest public techniques.
- **Seed:** the ProtoSSM path keeps seed `42` for diversity against the replica notebook.
- **BirdNET:** upgraded to the Model_9-style 3-second chunk branch, blended conservatively as a rank sidecar.
- **CV9245:** retained as an optional sidecar, but reduced to weight `0.03`.
- **Rare suppression:** saved as an ablation variant only, because it may be public-split tuned.


# 1. Environment, Wheels, and Global Config

Installs the offline wheels available in Kaggle inputs, locks CPU execution, seeds the run, and defines the submit-mode configuration.


In [ ]:
import subprocess, sys, os
from pathlib import Path
import random
import numpy as np
import torch

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(pattern)


def find_optional_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    return None

ONNX_WHL_CANDIDATES = [
    Path("/kaggle/input/datasets/nina2025/perch-v2-no-dft-v3-onnx/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"),
    Path("/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"),
    Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"),
]
ONNX_WHL = next((p for p in ONNX_WHL_CANDIDATES if p.exists()), None)
if ONNX_WHL is None:
    ONNX_WHL = find_optional_wheel("onnxruntime-1.24.4-*.whl")
if ONNX_WHL is not None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    print(f"ONNX Runtime installed from {ONNX_WHL}")
else:
    print("ONNX Runtime wheel not found; will try existing runtime or TF fallback")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
print("TF 2.20 installed")

try:
    import onnxruntime as ort
    _ONNX_AVAILABLE = True
    print("ONNX Runtime available")
except ImportError:
    _ONNX_AVAILABLE = False
    print("ONNX not available, falling back to TF")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print("Global random seed set to 42")

MODE = "submit"
assert MODE in {"train", "submit"}
print("MODE =", MODE)

import os, re, gc, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import pandas as pd
import soundfile as sf
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

_TF_MODULE = None

def load_tf():
    global _TF_MODULE
    if _TF_MODULE is None:
        import tensorflow as tf
        tf.experimental.numpy.experimental_enable_numpy_behavior()
        try:
            tf.config.set_visible_devices([], "GPU")
        except Exception:
            pass
        _TF_MODULE = tf
    return _TF_MODULE

_WALL_START = time.time()

BASE      = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
WORK_DIR  = Path("/kaggle/working/cache")
WORK_DIR.mkdir(parents=True, exist_ok=True)

SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12

CFG = {
    "batch_files": 16,
    "oof_n_splits": 5   if MODE == "train" else 3,
    "dryrun_n_files": 20 if MODE == "train" else 0,
    "run_oof": MODE == "train",
    "verbose": MODE == "train",
    "proto_ssm_train": {
        "n_epochs":        80  if MODE == "train" else 40,
        "lr":              8e-4,
        "weight_decay":    1e-3,
        "val_ratio":       0.15,
        "patience":        20  if MODE == "train" else 8,
        "pos_weight_cap":  25.0,
        "distill_weight":  0.15,
        "proto_margin":    0.15,
        "label_smoothing": 0.03,
        "oof_n_splits":    5   if MODE == "train" else 3,
        "mixup_alpha":     0.4,
        "focal_gamma":     2.5,
        "swa_start_frac":  0.65,
        "swa_lr":          4e-4,
        "use_cosine_restart": True,
        "restart_period":  20,
    },
    "residual_ssm": {
        "d_model": 128, "d_state": 16, "n_ssm_layers": 2,
        "dropout": 0.1, "correction_weight": 0.35,
        "n_epochs": 40  if MODE == "train" else 20,
        "lr": 8e-4,
        "patience": 12  if MODE == "train" else 6,
    },
    "mlp_params": {
        "hidden_layer_sizes": (256, 128), "activation": "relu",
        "max_iter": 500  if MODE == "train" else 200,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 20  if MODE == "train" else 10,
        "random_state": 42,
        "learning_rate_init": 5e-4,
        "alpha": 0.005,
    },
}
print("CFG loaded")

# 2. Labels, Metadata, and Perch Mapping

Builds the 234-class target matrix for fully labeled train soundscapes and maps competition labels onto Perch logits, including genus-level proxy handling for unmapped taxa.


In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
taxonomy          = pd.read_csv(BASE / "taxonomy.csv")
sample_sub        = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")
_dupe_label_rows = int(soundscape_labels.duplicated().sum())
if _dupe_label_rows:
    soundscape_labels = soundscape_labels.drop_duplicates().reset_index(drop=True)
    print(f"Dropped duplicated train_soundscapes_labels rows: {_dupe_label_rows}")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t: out.add(t)
    return sorted(out)

sc = (soundscape_labels
      .groupby(["filename", "start", "end"])["primary_label"]
      .apply(union_labels)
      .reset_index(name="label_list"))

sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"]  = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)

_meta = sc["filename"].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc["fully_labeled"] = sc["filename"].isin(full_files)

full_rows = (sc[sc["fully_labeled"]]
             .sort_values(["filename", "end_sec"])
             .reset_index(drop=False))
Y_FULL = Y_SC[full_rows["index"].to_numpy()]

print(f"Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}")
print(f"Full-file windows: {len(full_rows)} | Active classes: {int((Y_FULL.sum(0) > 0).sum())}")

# ── Perch backbone ────────────────────────────────────────────────────────────
# Prefer no-DFT variant, fallback to standard
ONNX_PERCH_PATH = next(INPUT_ROOT.glob("**/perch_v2_no_dft*.onnx"),
                   next(INPUT_ROOT.glob("**/perch_v2*.onnx"), Path("")))
USE_ONNX = _ONNX_AVAILABLE and ONNX_PERCH_PATH.exists()

infer_fn = None
if USE_ONNX:
    _so = ort.SessionOptions()
    _so.intra_op_num_threads = 4
    ONNX_SESSION    = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so,
                                            providers=["CPUExecutionProvider"])
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    print(f"Using ONNX Perch: {ONNX_PERCH_PATH.name}")
else:
    tf = load_tf()
    birdclassifier = tf.saved_model.load(str(MODEL_DIR))
    infer_fn       = birdclassifier.signatures["serving_default"]
    print("Using TF SavedModel Perch")

bc_labels = (pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
             .reset_index()
             .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}))
NO_LABEL = len(bc_labels)

mapping = (taxonomy
           .merge(bc_labels.rename(columns={"scientific_name": "scientific_name"}),
                  on="scientific_name", how="left"))
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit")

import re as _re
UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}

proxy_map = {}
unmapped_df = (taxonomy[taxonomy["primary_label"]
               .isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].copy())

for _, row in unmapped_df.iterrows():
    target = row["primary_label"]
    sci    = str(row["scientific_name"])
    genus  = sci.split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"]
        .astype(str)
        .str.match(rf"^{_re.escape(genus)}\s", na=False)
    ]
    if len(hits) > 0:
        proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
proxy_map  = {
    idx: bc_idxs
    for idx, bc_idxs in proxy_map.items()
    if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
}

print(f"Unmapped: {len(UNMAPPED_POS)} | Proxy: {len(proxy_map)} | No signal: {len(UNMAPPED_POS)-len(proxy_map)}")

# ── Per-taxon temperatures ────────────────────────────────────────────────────
temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, label in enumerate(PRIMARY_LABELS):
    cls = CLASS_NAME_MAP.get(label, "Aves")
    temperatures[ci] = 0.95 if cls in TEXTURE_TAXA else 1.10

# 3. Perch Inference and Training Cache

Defines the 60-second windowing path, runs ONNX Perch on hidden test soundscapes, and loads cached train Perch logits/embeddings for the small models trained at submit time.


In [ ]:
# ── Perch inference engine ────────────────────────────────────────────────────
import concurrent.futures

def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:                      y = y[:FILE_SAMPLES]
    return y

def run_perch(paths, batch_files=16, verbose=True):
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS
    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536),      dtype=np.float32)
    wr  = 0
    itr = tqdm(range(0, len(paths), batch_files), desc="Perch") if verbose else range(0, len(paths), batch_files)

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
        next_paths   = paths[0:batch_files]
        future_audio = [io_executor.submit(read_60s, p) for p in next_paths]
        for start in itr:
            batch_paths  = next_paths
            batch_n      = len(batch_paths)
            batch_audio  = [f.result() for f in future_audio]
            next_start = start + batch_files
            if next_start < len(paths):
                next_paths   = paths[next_start:next_start + batch_files]
                future_audio = [io_executor.submit(read_60s, p) for p in next_paths]
            x  = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = wr
            for bi, path in enumerate(batch_paths):
                y    = batch_audio[bi]
                meta = parse_fname(path.name)
                stem = path.stem
                x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids  [wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                filenames[wr:wr + N_WINDOWS] = path.name
                sites    [wr:wr + N_WINDOWS] = meta["site"]
                hours    [wr:wr + N_WINDOWS] = meta["hour_utc"]
                wr += N_WINDOWS
            if USE_ONNX:
                outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
                logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
                emb    = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
            else:
                tf = load_tf()
                out    = infer_fn(inputs=tf.convert_to_tensor(x))
                logits = out["label"].numpy().astype(np.float32)
                emb    = out["embedding"].numpy().astype(np.float32)
            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb
            for pos_idx, bc_idxs in proxy_map.items():
                bc_arr = np.array(bc_idxs, dtype=np.int32)
                scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)
            del x, logits, emb, batch_audio
            gc.collect()
    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames,
                             "site": sites, "hour_utc": hours})
    return meta_df, scores, embs

print("Perch inference engine defined")

# ── Cache ─────────────────────────────────────────────────────────────────────
print(f"USE_ONNX = {USE_ONNX}")

EXTERNAL_CACHE_DIRS = [
    Path("/kaggle/input/notebooks/vyankteshdwivedi/notebook1b25083f0d"),
    Path("/kaggle/input/datasets/jaejohn/perch-meta"),
]
CACHE_META_LOCAL = WORK_DIR / "perch_meta.parquet"
CACHE_NPZ_LOCAL  = WORK_DIR / "perch_arrays.npz"

def _find_external_cache():
    for d in EXTERNAL_CACHE_DIRS:
        meta = d / "perch_meta.parquet"
        npz  = d / "perch_arrays.npz"
        if meta.exists() and npz.exists():
            return meta, npz
    return None, None

SCORE_KEYS = ["scores", "sc", "logits", "perch_scores", "preds", "arr_0"]
EMB_KEYS   = ["embs", "emb", "embeddings", "features", "perch_embs", "arr_1"]

def _pick_array(arr, candidates, shape_hint_cols):
    for k in candidates:
        if k in arr.files:
            return arr[k], k
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == shape_hint_cols:
            return v, k
    raise KeyError(f"None of {candidates} found in npz. Available keys: {arr.files}")

def _build_cache():
    print(f"Building Perch cache from {len(full_files)} training files…")
    train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
    train_paths = [p for p in train_paths if p.exists()]
    t0 = time.time()
    meta_built, sc_built, emb_built = run_perch(train_paths, batch_files=CFG["batch_files"], verbose=True)
    print(f"  Perch pass done in {time.time()-t0:.1f}s  scores={sc_built.shape} embs={emb_built.shape}")
    meta_built.to_parquet(CACHE_META_LOCAL)
    np.savez(CACHE_NPZ_LOCAL, scores=sc_built.astype(np.float32),
             embs=emb_built.astype(np.float32), primary_labels=np.array(PRIMARY_LABELS))
    print(f"  Cache saved to {WORK_DIR}")
    return CACHE_META_LOCAL, CACHE_NPZ_LOCAL

ext_meta, ext_npz = _find_external_cache()
if ext_meta is not None:
    CACHE_META, CACHE_NPZ = ext_meta, ext_npz
    print(f"Using external cache: {CACHE_META.parent}")
elif CACHE_META_LOCAL.exists() and CACHE_NPZ_LOCAL.exists():
    CACHE_META, CACHE_NPZ = CACHE_META_LOCAL, CACHE_NPZ_LOCAL
    print(f"Using local cache: {WORK_DIR}")
else:
    print("No cache found — building from scratch")
    CACHE_META, CACHE_NPZ = _build_cache()

meta_tr = pd.read_parquet(CACHE_META)
_arr    = np.load(CACHE_NPZ)
sc_tr_raw,  sk = _pick_array(_arr, SCORE_KEYS, N_CLASSES)
emb_tr_raw, ek = _pick_array(_arr, EMB_KEYS,   1536)
sc_tr  = sc_tr_raw.astype(np.float32)
emb_tr = emb_tr_raw.astype(np.float32)

if "primary_labels" in _arr.files:
    if _arr["primary_labels"].tolist() != PRIMARY_LABELS:
        print("  WARNING: cached primary_labels differ — scores columns may not align!")
    else:
        print("  primary_labels schema OK")

if "row_id" not in meta_tr.columns:
    if "end_sec" in meta_tr.columns:
        end_sec = meta_tr["end_sec"].astype(int)
    elif "window_idx" in meta_tr.columns:
        end_sec = (meta_tr["window_idx"].astype(int) + 1) * 5
    else:
        end_sec = np.tile(np.arange(5, 65, 5), len(meta_tr) // N_WINDOWS)
    meta_tr["row_id"] = (meta_tr["filename"].str.replace(".ogg", "", regex=False)
                         + "_" + end_sec.astype(str))

row_id_to_index = full_rows.set_index("row_id")["index"]
missing_rows = set(meta_tr["row_id"]) - set(row_id_to_index.index)
if missing_rows:
    raise RuntimeError(f"Cache has {len(missing_rows)} row_ids not in labeled set.")

Y_FULL_aligned = Y_SC[row_id_to_index.loc[meta_tr["row_id"]].to_numpy()]
print(f"sc_tr: {sc_tr.shape}  emb_tr: {emb_tr.shape}  Y_FULL_aligned: {Y_FULL_aligned.shape}")

# 4. Priors, Probes, Calibration, and Post-processing Helpers

Adds site/hour priors, PCA + MLP probe utilities, threshold sharpening, file confidence scaling, and temporal smoothing helpers.


In [ ]:
# ── Post-processing helpers ───────────────────────────────────────────────────
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")

def smooth_predictions(probs, n_windows=12, alpha=0.3):
    N, C = probs.shape
    assert N % n_windows == 0
    view = probs.reshape(-1, n_windows, C).copy()
    prev_w = np.concatenate([view[:, :1, :],  view[:, :-1, :]], axis=1)
    next_w = np.concatenate([view[:, 1:,  :], view[:, -1:, :]], axis=1)
    return ((1 - alpha) * view + 0.5 * alpha * (prev_w + next_w)).reshape(N, C)

# ── UPGRADED prior tables — joint site-hour bucket ────────────────────────────
def build_prior_tables(sc_df, Y_labels):
    sc_df = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)

    site_keys = sorted(sc_df["site"].dropna().astype(str).unique())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_p = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]
        mask = sc_df["site"].astype(str).values == s
        site_n[i] = mask.sum()
        site_p[i] = Y_labels[mask].mean(axis=0)

    hour_keys = sorted(sc_df["hour_utc"].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]
        mask = sc_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum()
        hour_p[i] = Y_labels[mask].mean(axis=0)

    # Joint site-hour bucket (new — tighter shrinkage factor 4)
    sh_keys = sorted({(str(s), int(h)) for s, h in zip(sc_df["site"].dropna(), sc_df["hour_utc"].dropna())
                      if not pd.isna(s) and not pd.isna(h)})
    sh_to_i = {k: i for i, k in enumerate(sh_keys)}
    sh_p = np.zeros((len(sh_keys), Y_labels.shape[1]), dtype=np.float32)
    sh_n = np.zeros(len(sh_keys), dtype=np.float32)
    for (s, h) in sh_keys:
        i = sh_to_i[(s, h)]
        mask = (sc_df["site"].astype(str).values == s) & (sc_df["hour_utc"].astype(int).values == h)
        sh_n[i] = mask.sum()
        sh_p[i] = Y_labels[mask].mean(axis=0)

    return {
        "global_p": global_p,
        "site_to_i": site_to_i, "site_p": site_p, "site_n": site_n,
        "hour_to_i": hour_to_i, "hour_p": hour_p, "hour_n": hour_n,
        "sh_to_i": sh_to_i,    "sh_p": sh_p,    "sh_n": sh_n,
    }

def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
    eps = 1e-4; n = len(scores); out = scores.copy()
    p = np.tile(tables["global_p"], (n, 1))
    for i, h in enumerate(hours):
        h = int(h)
        if h in tables["hour_to_i"]:
            j = tables["hour_to_i"][h]; nh = tables["hour_n"][j]; w = nh / (nh + 8.0)
            p[i] = w * tables["hour_p"][j] + (1 - w) * tables["global_p"]
    for i, s in enumerate(sites):
        s = str(s)
        if s in tables["site_to_i"]:
            j = tables["site_to_i"][s]; ns = tables["site_n"][j]; w = ns / (ns + 8.0)
            p[i] = w * tables["site_p"][j] + (1 - w) * p[i]
    if "sh_to_i" in tables:
        for i, (s, h) in enumerate(zip(sites, hours)):
            key = (str(s), int(h))
            if key in tables["sh_to_i"]:
                j = tables["sh_to_i"][key]; nsh = tables["sh_n"][j]; w = nsh / (nsh + 4.0)
                p[i] = w * tables["sh_p"][j] + (1 - w) * p[i]
    p = np.clip(p, eps, 1 - eps)
    out += lambda_prior * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)

def file_confidence_scale(probs, n_windows=12, top_k=2, power=0.4):
    N, C = probs.shape
    view      = probs.reshape(-1, n_windows, C)
    sorted_v  = np.sort(view, axis=1)
    top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)
    return (view * np.power(top_k_mean, power)).reshape(N, C)

def rank_aware_scaling(probs, n_windows=12, power=0.4):
    N, C = probs.shape
    view     = probs.reshape(-1, n_windows, C)
    file_max = view.max(axis=1, keepdims=True)
    return (view * np.power(file_max, power)).reshape(N, C)

def adaptive_delta_smooth(probs, n_windows=12, base_alpha=0.20):
    N, C = probs.shape
    result = probs.copy(); view = probs.reshape(-1, n_windows, C); out = result.reshape(-1, n_windows, C)
    for t in range(n_windows):
        conf = view[:, t, :].max(axis=-1, keepdims=True); alpha = base_alpha * (1.0 - conf)
        if t == 0:           neighbor_avg = (view[:, t, :] + view[:, t+1, :]) / 2.0
        elif t == n_windows-1: neighbor_avg = (view[:, t-1, :] + view[:, t, :]) / 2.0
        else:                  neighbor_avg = (view[:, t-1, :] + view[:, t+1, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * neighbor_avg
    return result

# ── MLP probes ────────────────────────────────────────────────────────────────
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
import torch.nn as nn
import torch.nn.functional as F

def build_class_freq_weights(Y, cap=10.0):
    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    freq = pos_count / Y.shape[0]
    weights = np.clip(1.0 / (freq ** 0.5), 1.0, cap)
    return (weights / weights.mean()).astype(np.float32)

def build_sequential_features(scores_col, n_windows=12):
    x     = scores_col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
    next_ = np.concatenate([x[:, 1:], x[:, -1:]], axis=1)
    mean  = np.repeat(x.mean(axis=1), n_windows)
    max_  = np.repeat(x.max(axis=1),  n_windows)
    std   = np.repeat(x.std(axis=1),  n_windows)
    return prev.reshape(-1), next_.reshape(-1), mean, max_, std

def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.4):
    scaler = StandardScaler(); emb_s = scaler.fit_transform(emb)
    pca    = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z      = pca.fit_transform(emb_s).astype(np.float32)
    print(f"Embedding: {emb.shape} → PCA: {Z.shape}  (variance retained: {pca.explained_variance_ratio_.sum():.2%})")
    class_weights = build_class_freq_weights(Y, cap=10.0)
    probe_models = {}; active = np.where(Y.sum(axis=0) >= min_pos)[0]; MAX_ROWS = 3000
    for ci in tqdm(active, desc="MLP probes"):
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y): continue
        prev, next_, mean, max_, std = build_sequential_features(scores_raw[:, ci])
        X = np.hstack([Z, scores_raw[:, ci:ci+1], prev[:, None], next_[:, None], mean[:, None], max_[:, None], std[:, None]])
        n_pos = int(y.sum()); n_neg = len(y) - n_pos; pos_idx = np.where(y == 1)[0]
        w = float(class_weights[ci]); repeat = max(1, min(int(round(w * n_neg / max(n_pos, 1))), 8))
        if n_pos * repeat + len(y) > MAX_ROWS: repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))
        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])
        clf = MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", max_iter=300,
                            early_stopping=True, validation_fraction=0.15, n_iter_no_change=15,
                            random_state=42, learning_rate_init=5e-4, alpha=0.005)
        clf.fit(X_bal, y_bal); probe_models[ci] = clf
    print(f"Trained {len(probe_models)} MLP probes")
    return probe_models, scaler, pca, alpha_blend

class VectorizedMLPProbes(nn.Module):
    def __init__(self, probe_models):
        super().__init__(); self.valid_classes = sorted(probe_models.keys()); V = len(self.valid_classes)
        if V == 0: self.weights = nn.ParameterList(); self.biases = nn.ParameterList(); self.n_layers = 0; return
        sample = probe_models[self.valid_classes[0]]; self.n_layers = len(sample.coefs_)
        self.weights = nn.ParameterList(); self.biases = nn.ParameterList()
        for li in range(self.n_layers):
            W = np.stack([probe_models[c].coefs_[li] for c in self.valid_classes], axis=0)
            b = np.stack([probe_models[c].intercepts_[li] for c in self.valid_classes], axis=0)
            self.weights.append(nn.Parameter(torch.tensor(W, dtype=torch.float32), requires_grad=False))
            self.biases.append(nn.Parameter(torch.tensor(b, dtype=torch.float32), requires_grad=False))
    def forward(self, x):
        h = x
        for i in range(self.n_layers):
            h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
            if i < self.n_layers - 1: h = torch.relu(h)
        return h.squeeze(-1)

def apply_mlp_probes_vectorized(emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.4):
    if len(probe_models) == 0: return scores_test.copy()
    Z_test = pca.transform(scaler.transform(emb_test)).astype(np.float32)
    valid_classes = sorted(probe_models.keys()); V = len(valid_classes); N = len(scores_test)
    raw = scores_test[:, valid_classes].T; n_files = N // N_WINDOWS; raw_view = raw.reshape(V, n_files, N_WINDOWS)
    prev = np.concatenate([raw_view[:, :, :1], raw_view[:, :, :-1]], axis=2).reshape(V, N)
    nxt  = np.concatenate([raw_view[:, :, 1:], raw_view[:, :, -1:]], axis=2).reshape(V, N)
    mean = np.repeat(raw_view.mean(axis=2), N_WINDOWS, axis=1); mx = np.repeat(raw_view.max(axis=2), N_WINDOWS, axis=1)
    std  = np.repeat(raw_view.std(axis=2),  N_WINDOWS, axis=1)
    scalar_feats = np.stack([raw, prev, nxt, mean, mx, std], axis=-1).astype(np.float32)
    Z_expanded = np.broadcast_to(Z_test, (V, N, Z_test.shape[1]))
    X_all = np.concatenate([Z_expanded.astype(np.float32), scalar_feats], axis=-1)
    vec_probe = VectorizedMLPProbes(probe_models).eval()
    with torch.no_grad(): preds = vec_probe(torch.tensor(X_all)).numpy()
    result = scores_test.copy()
    result[:, valid_classes] = (1.0 - alpha_blend) * scores_test[:, valid_classes] + alpha_blend * preds.T
    return result

def calibrate_and_optimize_thresholds(oof_probs, Y_FULL, threshold_grid=None, n_windows=12):
    if threshold_grid is None: threshold_grid = [0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]
    n_samples, n_cls = oof_probs.shape; thresholds = np.full(n_cls, 0.5, dtype=np.float32)
    n_files = n_samples // n_windows
    file_oof = oof_probs.reshape(n_files, n_windows, n_cls).max(axis=1)
    file_y   = Y_FULL.reshape(n_files, n_windows, n_cls).max(axis=1)
    n_calibrated = 0
    for c in range(n_cls):
        y_true = file_y[:, c]; y_prob = file_oof[:, c]
        if y_true.sum() < 3: continue
        try:
            ir = IsotonicRegression(out_of_bounds="clip"); ir.fit(y_prob, y_true); y_cal = ir.transform(y_prob)
        except: y_cal = y_prob
        best_f1, best_t = 0.0, 0.5
        for t in threshold_grid:
            pred = (y_cal >= t).astype(int)
            tp=((pred==1)&(y_true==1)).sum(); fp=((pred==1)&(y_true==0)).sum(); fn=((pred==0)&(y_true==1)).sum()
            prec=tp/(tp+fp+1e-8); rec=tp/(tp+fn+1e-8); f1=2*prec*rec/(prec+rec+1e-8)
            if f1 > best_f1: best_f1,best_t = f1,t
        thresholds[c] = best_t; n_calibrated += 1
    print(f"Calibrated {n_calibrated} classes | Mean threshold: {thresholds.mean():.3f} | Range: [{thresholds.min():.2f}, {thresholds.max():.2f}]")
    return thresholds

def apply_per_class_thresholds(scores, thresholds):
    C = scores.shape[1]; scaled = np.copy(scores)
    for c in range(C):
        t = thresholds[c]; above = scores[:, c] > t
        scaled[above, c]  = 0.5 + 0.5 * (scores[above, c]  - t) / (1 - t + 1e-8)
        scaled[~above, c] = 0.5 * scores[~above, c] / (t + 1e-8)
    return np.clip(scaled, 0.0, 1.0)

# 5. Sequence Models

Defines the lightweight bidirectional SSM blocks, ProtoSSM head, and ResidualSSM correction model.


In [ ]:
# ── SSM Architecture ─────────────────────────────────────────────────────────
class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__(); self.d_model=d_model; self.d_state=d_state
        self.in_proj=nn.Linear(d_model,2*d_model,bias=False)
        self.conv1d=nn.Conv1d(d_model,d_model,d_conv,padding=d_conv-1,groups=d_model)
        self.dt_proj=nn.Linear(d_model,d_model,bias=True)
        A=torch.arange(1,d_state+1,dtype=torch.float32).unsqueeze(0).expand(d_model,-1)
        self.A_log=nn.Parameter(torch.log(A)); self.D=nn.Parameter(torch.ones(d_model))
        self.B_proj=nn.Linear(d_model,d_state,bias=False); self.C_proj=nn.Linear(d_model,d_state,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
    def forward(self,x):
        B_sz,T,D=x.shape; xz=self.in_proj(x); x_ssm,z=xz.chunk(2,dim=-1)
        x_conv=F.silu(self.conv1d(x_ssm.transpose(1,2))[:,:,:T].transpose(1,2))
        dt=F.softplus(self.dt_proj(x_conv)); A=-torch.exp(self.A_log)
        B=self.B_proj(x_conv); C=self.C_proj(x_conv)
        h=torch.zeros(B_sz,D,self.d_state,device=x.device); ys=[]
        for t in range(T):
            dA=torch.exp(A[None]*dt[:,t,:,None]); dB=dt[:,t,:,None]*B[:,t,None,:]
            h=h*dA+x[:,t,:,None]*dB; ys.append((h*C[:,t,None,:]).sum(-1))
        return torch.stack(ys,dim=1)+x*self.D[None,None,:]

class LightProtoSSM(nn.Module):
    def __init__(self,d_input=1536,d_model=128,d_state=16,n_classes=234,n_windows=12,
                 dropout=0.15,n_sites=20,meta_dim=16,use_cross_attn=True,cross_attn_heads=2):
        super().__init__(); self.n_classes=n_classes; self.n_windows=n_windows; self.use_cross_attn=use_cross_attn
        self.input_proj=nn.Sequential(nn.Linear(d_input,d_model),nn.LayerNorm(d_model),nn.GELU(),nn.Dropout(dropout))
        self.pos_enc=nn.Parameter(torch.randn(1,n_windows,d_model)*0.02)
        self.site_emb=nn.Embedding(n_sites,meta_dim); self.hour_emb=nn.Embedding(24,meta_dim)
        self.meta_proj=nn.Linear(2*meta_dim,d_model)
        self.ssm_fwd=nn.ModuleList([SelectiveSSM(d_model,d_state) for _ in range(2)])
        self.ssm_bwd=nn.ModuleList([SelectiveSSM(d_model,d_state) for _ in range(2)])
        self.ssm_merge=nn.ModuleList([nn.Linear(2*d_model,d_model) for _ in range(2)])
        self.ssm_norm=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.drop=nn.Dropout(dropout)
        if use_cross_attn:
            self.cross_attn=nn.ModuleList([nn.MultiheadAttention(d_model,cross_attn_heads,dropout=dropout,batch_first=True) for _ in range(2)])
            self.cross_norm=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.prototypes=nn.Parameter(torch.randn(n_classes,d_model)*0.02)
        self.proto_temp=nn.Parameter(torch.tensor(5.0))
        self.class_bias=nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha=nn.Parameter(torch.zeros(n_classes))
    def init_prototypes(self,emb_tensor,labels_tensor):
        with torch.no_grad():
            h=self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask=labels_tensor[:,c]>0.5
                if mask.sum()>0: self.prototypes.data[c]=F.normalize(h[mask].mean(0),dim=0)
    def forward(self,emb,perch_logits=None,site_ids=None,hours=None):
        B,T,_=emb.shape; h=self.input_proj(emb)+self.pos_enc[:,:T,:]
        if site_ids is not None and hours is not None:
            meta=self.meta_proj(torch.cat([self.site_emb(site_ids),self.hour_emb(hours)],dim=-1))
            h=h+meta[:,None,:]
        for i,(fwd,bwd,merge,norm) in enumerate(zip(self.ssm_fwd,self.ssm_bwd,self.ssm_merge,self.ssm_norm)):
            res=h; hf=fwd(h); hb=bwd(h.flip(1)).flip(1)
            h=self.drop(merge(torch.cat([hf,hb],dim=-1))); h=norm(h+res)
            if self.use_cross_attn:
                attn_out,_=self.cross_attn[i](h,h,h); h=self.cross_norm[i](h+attn_out)
        h_n=F.normalize(h,dim=-1); p_n=F.normalize(self.prototypes,dim=-1)
        sim=torch.matmul(h_n,p_n.T)*F.softplus(self.proto_temp)+self.class_bias[None,None,:]
        if perch_logits is not None:
            alpha=torch.sigmoid(self.fusion_alpha)[None,None,:]
            out=alpha*sim+(1-alpha)*perch_logits
        else: out=sim
        return out

class ResidualSSM(nn.Module):
    def __init__(self,d_input=1536,d_scores=234,d_model=64,d_state=8,n_classes=234,
                 n_windows=12,dropout=0.1,n_sites=20,meta_dim=8):
        super().__init__(); self.n_classes=n_classes
        self.input_proj=nn.Sequential(nn.Linear(d_input+d_scores,d_model),nn.LayerNorm(d_model),nn.GELU(),nn.Dropout(dropout))
        self.site_emb=nn.Embedding(n_sites,meta_dim); self.hour_emb=nn.Embedding(24,meta_dim)
        self.meta_proj=nn.Linear(2*meta_dim,d_model)
        self.pos_enc=nn.Parameter(torch.randn(1,n_windows,d_model)*0.02)
        self.ssm_fwd=SelectiveSSM(d_model,d_state); self.ssm_bwd=SelectiveSSM(d_model,d_state)
        self.ssm_merge=nn.Linear(2*d_model,d_model); self.ssm_norm=nn.LayerNorm(d_model); self.ssm_drop=nn.Dropout(dropout)
        self.output_head=nn.Linear(d_model,n_classes)
        nn.init.zeros_(self.output_head.weight); nn.init.zeros_(self.output_head.bias)
    def forward(self,emb,first_pass,site_ids=None,hours=None):
        B,T,_=emb.shape; x=torch.cat([emb,first_pass],dim=-1)
        h=self.input_proj(x)+self.pos_enc[:,:T,:]
        if site_ids is not None and hours is not None:
            meta=self.meta_proj(torch.cat([self.site_emb(site_ids.clamp(0,self.site_emb.num_embeddings-1)),
                                            self.hour_emb(hours.clamp(0,23))],dim=-1))
            h=h+meta.unsqueeze(1)
        res=h; hf=self.ssm_fwd(h); hb=self.ssm_bwd(h.flip(1)).flip(1)
        h=self.ssm_drop(self.ssm_merge(torch.cat([hf,hb],dim=-1))); h=self.ssm_norm(h+res)
        return self.output_head(h)

def train_light_proto_ssm(emb_full, scores_full, Y_full, meta_full, n_epochs=40, patience=8, lr=1e-3, n_sites=20, verbose=False):
    n_files=len(emb_full)//N_WINDOWS; emb_f=emb_full.reshape(n_files,N_WINDOWS,-1)
    log_f=scores_full.reshape(n_files,N_WINDOWS,-1); lab_f=Y_full.reshape(n_files,N_WINDOWS,-1).astype(np.float32)
    fnames=meta_full["filename"].unique(); sites_u=sorted(meta_full["site"].unique())
    site2i={s:i+1 for i,s in enumerate(sites_u)}
    site_ids=np.array([min(site2i.get(meta_full.loc[meta_full["filename"]==fn,"site"].iloc[0],0),n_sites-1) for fn in fnames],dtype=np.int64)
    hour_ids=np.array([int(meta_full.loc[meta_full["filename"]==fn,"hour_utc"].iloc[0])%24 for fn in fnames],dtype=np.int64)
    model=LightProtoSSM(n_classes=N_CLASSES,n_sites=n_sites,use_cross_attn=True,cross_attn_heads=2)
    model.init_prototypes(torch.tensor(emb_full,dtype=torch.float32),torch.tensor(Y_full,dtype=torch.float32))
    emb_t=torch.tensor(emb_f,dtype=torch.float32); log_t=torch.tensor(log_f,dtype=torch.float32)
    lab_t=torch.tensor(lab_f,dtype=torch.float32); site_t=torch.tensor(site_ids,dtype=torch.long)
    hour_t=torch.tensor(hour_ids,dtype=torch.long)
    pos_cnt=lab_t.sum(dim=(0,1)); total=lab_t.shape[0]*lab_t.shape[1]
    pos_weight=((total-pos_cnt)/(pos_cnt+1)).clamp(max=25.0)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-3)
    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,epochs=n_epochs,steps_per_epoch=1,pct_start=0.1,anneal_strategy="cos")
    best_loss,best_state,wait=float("inf"),None,0
    swa_model=torch.optim.swa_utils.AveragedModel(model); swa_start=int(n_epochs*0.65)
    swa_sched=torch.optim.swa_utils.SWALR(opt,swa_lr=4e-4)
    for ep in range(n_epochs):
        model.train()
        out=model(emb_t,log_t,site_ids=site_t,hours=hour_t)
        loss=F.binary_cross_entropy_with_logits(out,lab_t,pos_weight=pos_weight[None,None,:])+0.15*F.mse_loss(out,log_t)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        if ep>=swa_start: swa_model.update_parameters(model); swa_sched.step()
        else: sched.step()
        if loss.item()<best_loss:
            best_loss=loss.item(); best_state={k:v.clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: break
    if ep>=swa_start:
        torch.optim.swa_utils.update_bn(emb_t.unsqueeze(0),swa_model); model=swa_model
    else: model.load_state_dict(best_state)
    model.eval(); return model,site2i

def run_tta_proto(proto_model, emb_files, sc_files, site_t, hour_t, shifts=[0,1,-1,2,-2]):
    proto_model.eval(); all_preds=[]
    emb_t=torch.tensor(emb_files,dtype=torch.float32); sc_t=torch.tensor(sc_files,dtype=torch.float32)
    for shift in shifts:
        e=torch.roll(emb_t,shift,dims=1) if shift else emb_t
        s=torch.roll(sc_t,shift,dims=1) if shift else sc_t
        with torch.no_grad():
            out=proto_model(e,s,site_ids=site_t,hours=hour_t).numpy()
        if shift: out=np.roll(out,-shift,axis=1)
        all_preds.append(out)
    return np.mean(all_preds,axis=0)

def train_residual_ssm(emb_full, first_pass_flat, Y_full, site_ids, hour_ids,
                        n_epochs=30, patience=8, lr=1e-3, correction_weight=0.30, verbose=False):
    n_files=len(emb_full)//N_WINDOWS; emb_f=emb_full.reshape(n_files,N_WINDOWS,-1)
    fp_f=first_pass_flat.reshape(n_files,N_WINDOWS,-1); lab_f=Y_full.reshape(n_files,N_WINDOWS,-1).astype(np.float32)
    fp_prob=1.0/(1.0+np.exp(-np.clip(fp_f,-30,30))); residuals=lab_f-fp_prob
    n_val=max(1,int(n_files*0.15)); rng=torch.Generator(); rng.manual_seed(42)
    perm=torch.randperm(n_files,generator=rng).numpy(); val_i=perm[:n_val]; train_i=perm[n_val:]
    emb_t=torch.tensor(emb_f,dtype=torch.float32); fp_t=torch.tensor(fp_f,dtype=torch.float32)
    res_t=torch.tensor(residuals,dtype=torch.float32)
    site_t=torch.tensor(site_ids,dtype=torch.long); hour_t=torch.tensor(hour_ids,dtype=torch.long)
    model=ResidualSSM(n_classes=N_CLASSES)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-3)
    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,epochs=n_epochs,steps_per_epoch=1,pct_start=0.1,anneal_strategy="cos")
    best_loss,best_state,wait=float("inf"),None,0
    for ep in range(n_epochs):
        model.train()
        corr=model(emb_t[train_i],fp_t[train_i],site_ids=site_t[train_i],hours=hour_t[train_i])
        loss=F.mse_loss(corr,res_t[train_i])
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            val_corr=model(emb_t[val_i],fp_t[val_i],site_ids=site_t[val_i],hours=hour_t[val_i])
            val_loss=F.mse_loss(val_corr,res_t[val_i])
        if val_loss.item()<best_loss:
            best_loss=val_loss.item(); best_state={k:v.clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: break
    model.load_state_dict(best_state); return model,correction_weight

print("Sequence Models defined")

# 6. ProtoSSM Branch

Runs Perch on the hidden test files, trains the lightweight sequence/probe stack on the labeled train soundscapes, and writes `submission_protossm.csv`.


In [ ]:
# ── Test inference ────────────────────────────────────────────────────────────
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
if IS_DRY_RUN:
    n = CFG["dryrun_n_files"] or 20
    print(f"No hidden test — dry-run on {n} train files")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:n]
else:
    print(f"Hidden test files: {len(test_paths)}")

meta_te, sc_te, emb_te = run_perch(test_paths, CFG["batch_files"], verbose=CFG["verbose"])
print(f"Test scores: {sc_te.shape}")

# ── Full ProtoSSM pipeline ────────────────────────────────────────────────────
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

t0 = time.time()
proto_model, site2i_tr = train_light_proto_ssm(
    emb_tr, sc_tr, Y_FULL_aligned, meta_tr,
    n_epochs=40, patience=8, lr=1e-3, verbose=False)
print(f"ProtoSSM training: {time.time()-t0:.1f}s")

n_test_files  = len(sc_te) // N_WINDOWS
emb_te_f      = emb_te.reshape(n_test_files, N_WINDOWS, -1)
sc_te_f       = sc_te.reshape(n_test_files, N_WINDOWS, -1)

test_fnames   = meta_te.drop_duplicates("filename")["filename"].tolist()
n_sites_cap   = 20
test_site_ids = np.array([min(site2i_tr.get(meta_te.loc[meta_te["filename"]==fn,"site"].iloc[0],0),n_sites_cap-1)
                           for fn in test_fnames], dtype=np.int64)
test_hour_ids = np.array([int(meta_te.loc[meta_te["filename"]==fn,"hour_utc"].iloc[0])%24
                           for fn in test_fnames], dtype=np.int64)

proto_model.eval()
with torch.no_grad():
    proto_out = proto_model(
        torch.tensor(emb_te_f, dtype=torch.float32),
        torch.tensor(sc_te_f,  dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours   =torch.tensor(test_hour_ids, dtype=torch.long),
    ).numpy()
proto_scores_flat = proto_out.reshape(-1, N_CLASSES).astype(np.float32)

prior_tables   = build_prior_tables(sc, Y_SC)
sc_te_adjusted = apply_prior(sc_te, sites=meta_te["site"].to_numpy(),
                              hours=meta_te["hour_utc"].to_numpy(), tables=prior_tables, lambda_prior=0.4)

probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
    emb=emb_tr, scores_raw=sc_tr, Y=Y_FULL_aligned, min_pos=5, pca_dim=64, alpha_blend=0.4)
sc_te_adjusted = apply_mlp_probes_vectorized(emb_te, sc_te_adjusted, probe_models, emb_scaler, emb_pca, alpha_blend)

ENSEMBLE_W      = 0.5
first_pass_flat = (ENSEMBLE_W * proto_scores_flat + (1.0 - ENSEMBLE_W) * sc_te_adjusted)

n_tr_files    = len(sc_tr) // N_WINDOWS
emb_tr_f      = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
sc_tr_f       = sc_tr.reshape(n_tr_files, N_WINDOWS, -1)

tr_fnames     = meta_tr.drop_duplicates("filename")["filename"].tolist()
tr_site_ids   = np.array([min(site2i_tr.get(meta_tr.loc[meta_tr["filename"]==fn,"site"].iloc[0],0),n_sites_cap-1)
                           for fn in tr_fnames], dtype=np.int64)
tr_hour_ids   = np.array([int(meta_tr.loc[meta_tr["filename"]==fn,"hour_utc"].iloc[0])%24
                           for fn in tr_fnames], dtype=np.int64)

proto_tr_out = run_tta_proto(proto_model, emb_tr_f, sc_tr_f,
    site_t=torch.tensor(tr_site_ids, dtype=torch.long),
    hour_t=torch.tensor(tr_hour_ids, dtype=torch.long),
    shifts=[0, 1, -1, 2, -2])
proto_tr_flat = proto_tr_out.reshape(-1, N_CLASSES).astype(np.float32)

sc_tr_prior = apply_prior(sc_tr, sites=meta_tr["site"].to_numpy(),
                           hours=meta_tr["hour_utc"].to_numpy(), tables=prior_tables, lambda_prior=0.4)
sc_tr_mlp = apply_mlp_probes_vectorized(emb_tr, sc_tr_prior, probe_models, emb_scaler, emb_pca, alpha_blend)
first_pass_tr = (ENSEMBLE_W * proto_tr_flat + (1.0 - ENSEMBLE_W) * sc_tr_mlp)

train_probs_for_calib = sigmoid(first_pass_tr)
PER_CLASS_THRESHOLDS = calibrate_and_optimize_thresholds(
    oof_probs=train_probs_for_calib, Y_FULL=Y_FULL_aligned,
    threshold_grid=[0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70], n_windows=N_WINDOWS)

t0 = time.time()
res_model, correction_weight = train_residual_ssm(
    emb_full=emb_tr, first_pass_flat=first_pass_tr, Y_full=Y_FULL_aligned,
    site_ids=tr_site_ids, hour_ids=tr_hour_ids, n_epochs=30, patience=8, lr=1e-3,
    correction_weight=0.30, verbose=False)
print(f"ResidualSSM training: {time.time()-t0:.1f}s")

first_pass_te_f = first_pass_flat.reshape(n_test_files, N_WINDOWS, -1)
res_model.eval()
with torch.no_grad():
    test_correction = res_model(
        torch.tensor(emb_te_f,         dtype=torch.float32),
        torch.tensor(first_pass_te_f,  dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours   =torch.tensor(test_hour_ids, dtype=torch.long),
    ).numpy()
correction_flat = test_correction.reshape(-1, N_CLASSES).astype(np.float32)
final_scores    = first_pass_flat + correction_weight * correction_flat
final_scores    = final_scores / temperatures[None, :]
probs = sigmoid(final_scores)
probs = file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=0.4)
probs = rank_aware_scaling(probs,    n_windows=N_WINDOWS, power=0.4)
probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=0.20)
probs = np.clip(probs, 0.0, 1.0)
probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS)   # ← now applied

sub = pd.DataFrame(probs.astype(np.float32), columns=PRIMARY_LABELS)
sub.insert(0, "row_id", meta_te["row_id"].values)
sub.to_csv("submission_protossm.csv", index=False)
print("ProtoSSM execution complete")
print(f"Total wall time so far: {(time.time() - _WALL_START)/60:.1f} min")
del emb_tr_f, sc_tr_f, proto_model, res_model
gc.collect()
print("Memory freed. Ready for SED cell.")

# 7. Distilled SED Branch

Runs the public Tucker Arrants distilled SED ONNX folds and writes `submission_sed.csv`.


In [ ]:
import librosa
from scipy.ndimage import gaussian_filter1d

N_MELS_SED = 256
N_FFT_SED  = 2048
HOP_SED    = 512
FMIN_SED   = 20
FMAX_SED   = 16000
TOP_DB_SED = 80

def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        raise FileNotFoundError("sed_fold0.onnx not found. Attach tuckerarrants/bc2026-distilled-sed-public.")
    return hits[0].parent

def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so, providers=["CPUExecutionProvider"])

def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
                                            n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0)
        s = librosa.power_to_db(s, top_db=TOP_DB_SED)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)

def file_to_sed_chunks(path):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if sr0 != SR: y = librosa.resample(y, orig_sr=sr0, target_sr=SR)
    n = 60 * SR
    if len(y) < n: y = np.pad(y, (0, n - len(y)))
    else:          y = y[:n]
    chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    ends   = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC
    return chunks, ends

def sigmoid_sed(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)

# Use the same test files as Cell 1
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
if IS_DRY_RUN:
    dry_n = CFG["dryrun_n_files"] if "CFG" in dir() else 20
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:(dry_n or 20)]

sed_dir = find_sed_dir()
sed_fold_paths = sorted(sed_dir.glob("sed_fold*.onnx"),
                         key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1)))
sed_sessions = [make_sed_session(p) for p in sed_fold_paths]

print(f"SED dir: {sed_dir}")
print(f"SED folds loaded: {[p.name for p in sed_fold_paths]}")

sed_rows, sed_preds = [], []

for i, path in enumerate(test_paths, 1):
    chunks, ends = file_to_sed_chunks(path)
    mel = audio_to_mel(chunks)
    p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)

    for sess in sed_sessions:
        outs = sess.run(None, {sess.get_inputs()[0].name: mel})
        clip_logits = outs[0]             # (12, 234)
        frame_max   = outs[1].max(axis=1) # (12, 234)
        p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)

    p_mean = p_sum / len(sed_sessions)

    if len(p_mean) > 1:
        p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)

    stem = path.stem
    sed_rows.extend([f"{stem}_{int(t)}" for t in ends])
    sed_preds.append(p_mean)

    if i == 1 or i % 50 == 0 or i == len(test_paths):
        print(f"SED: {i}/{len(test_paths)}")

sed_preds_arr = np.concatenate(sed_preds, axis=0)
sed_sub = pd.DataFrame(np.clip(sed_preds_arr, 0.0, 1.0), columns=PRIMARY_LABELS)
sed_sub.insert(0, "row_id", sed_rows)
sed_sub.to_csv("submission_sed.csv", index=False)
print(f"Distilled SED Processing Complete. Shape: {sed_sub.shape}")

# 8. Robust Hybrid Rank Ensemble, CV9245 Sidecar, and Stronger BirdNET

Combines the existing multi-view ProtoSSM/SED blend with optional CV9245 and a stronger Model_9-style BirdNET branch. Rare suppression is still written only as an ablation variant.


In [ ]:
import os
import importlib.util
import numpy as np
import pandas as pd
from pathlib import Path

PROTOSSM_CSV = "submission_protossm.csv"
SED_CSV      = "submission_sed.csv"
OUT_CSV      = "submission.csv"
EPS = 1e-5

CV9245_OUT = Path("submission_cv9245_cnnonly_sharedperch.csv")
CV9245_RANK_WEIGHT = 0.03
CV9245_START_CUTOFF_MIN = 65.0
CV9245_BATCH_FILES = 4
BIRDNET_OUT = Path("submission_birdnet_public.csv")
BIRDNET_RANK_WEIGHT = 0.15
BIRDNET_START_CUTOFF_MIN = 80.0
TOTAL_RUNTIME_GUARD_MIN = 86.0

BASE_PATH = Path("/kaggle/input/competitions/birdclef-2026")
df_proto = pd.read_csv(PROTOSSM_CSV)
df_sed   = pd.read_csv(SED_CSV)

cols = [c for c in df_proto.columns if c != "row_id"]

# Align row order.
df_sed = df_sed.set_index("row_id").loc[df_proto["row_id"]].reset_index()
p_proto = np.clip(df_proto[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
p_sed   = np.clip(df_sed[cols].to_numpy(np.float32),   EPS, 1.0 - EPS)

row_ids  = df_proto["row_id"].astype(str).to_numpy()
file_ids = np.array(["_".join(r.split("_")[:-1]) for r in row_ids])


def _elapsed_min():
    return (time.time() - _WALL_START) / 60.0


def _validate_prediction_frame(name, df, template_df, cols, fill_missing=False):
    if df is None:
        raise ValueError(f"{name}: empty prediction frame")
    if "row_id" not in df.columns:
        raise ValueError(f"{name}: missing row_id column")
    missing_cols = [c for c in cols if c not in df.columns]
    if missing_cols and not fill_missing:
        raise ValueError(f"{name}: missing {len(missing_cols)} class columns")

    aligned = df.set_index("row_id").reindex(template_df["row_id"].astype(str))
    if aligned.index.hasnans:
        raise ValueError(f"{name}: row_id alignment failed")
    if missing_cols and fill_missing:
        for c in missing_cols:
            aligned[c] = 0.0
    aligned = aligned[cols]
    if aligned.isna().all(axis=1).any():
        raise ValueError(f"{name}: one or more row_ids did not align")
    aligned = aligned.fillna(0.0)

    arr = aligned.to_numpy(np.float32)
    expected_shape = (len(template_df), len(cols))
    if arr.shape != expected_shape:
        raise ValueError(f"{name}: expected {expected_shape}, got {arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name}: non-finite predictions")
    return np.clip(arr, 0.0, 1.0).astype(np.float32)


def _blend_rank_branch(base_pred, branch_pred, weight, name, class_mask=None):
    if weight <= 0:
        return base_pred
    base_rank = _rank_columns(np.clip(base_pred, EPS, 1.0 - EPS))
    branch_rank = _rank_columns(np.clip(branch_pred, EPS, 1.0 - EPS))
    out = base_rank.copy()
    if class_mask is not None:
        class_mask = np.asarray(class_mask, dtype=bool)
        if class_mask.shape[0] != out.shape[1]:
            raise ValueError(f"{name}: class mask length mismatch")
        if not class_mask.any():
            print(f"{name} branch has no mapped classes; skipping")
            return base_pred
        out = base_pred.copy()
        out[:, class_mask] = ((1.0 - weight) * base_rank[:, class_mask]
                              + weight * branch_rank[:, class_mask])
    else:
        out = (1.0 - weight) * base_rank + weight * branch_rank
    print(f"Applied {name} rank branch weight={weight:.3f}")
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def _rank_columns(arr):
    return pd.DataFrame(arr).rank(axis=0, pct=True).to_numpy(np.float32)


def _base_rank_blend(p_proto_in, p_sed_in, proto_weight=0.60):
    rank_proto = _rank_columns(p_proto_in)
    rank_sed   = _rank_columns(p_sed_in)
    pred = (rank_proto * proto_weight) + (rank_sed * (1.0 - proto_weight))

    fake_only = (p_proto_in > 0.50) & (p_sed_in < 0.05)
    pred = np.where(fake_only, (1.0 - 0.08) * pred + 0.08 * rank_proto, pred)

    offs = np.arange(-3, 4, dtype=np.float32)
    proto_kernel = (1.0 + (offs / 1.20) ** 2 / 2.0) ** (-1.5)
    proto_kernel = (proto_kernel / proto_kernel.sum()).astype(np.float32)

    pa_ctx = p_proto_in.copy()
    for fid in pd.unique(file_ids):
        m = file_ids == fid
        x = p_proto_in[m]
        if len(x) > 1:
            xp = np.pad(x, ((3, 3), (0, 0)), mode="edge")
            pa_ctx[m] = sum(proto_kernel[i] * xp[i:i + len(x)] for i in range(7))

    xctx = _rank_columns(pa_ctx)
    proto_cont = (xctx > 0.88) & (rank_proto > 0.75) & (p_sed_in < 0.12) & (~fake_only)
    pred = np.where(proto_cont, (1.0 - 0.15) * pred + 0.15 * np.maximum(rank_proto, xctx), pred)

    sed_only = (rank_sed > 0.95) & (rank_proto < 0.80) & (~fake_only) & (~proto_cont)
    pred = np.where(sed_only, (1.0 - 0.12) * pred + 0.12 * rank_sed, pred)
    return np.clip(pred, 0.0, 1.0).astype(np.float32)


def _event_class_mask():
    mask = np.ones(len(cols), dtype=bool)
    for i, label in enumerate(cols):
        if CLASS_NAME_MAP.get(label, "Aves") in TEXTURE_TAXA:
            mask[i] = False
    return mask


def _local_max_blend(pred, alpha=0.10, radius=1):
    if alpha <= 0:
        return pred.copy()
    out = pred.copy()
    event_mask = _event_class_mask()
    for fid in pd.unique(file_ids):
        idx = np.where(file_ids == fid)[0]
        slab = pred[idx]
        prop = slab.copy()
        for t in range(len(idx)):
            lo = max(0, t - radius)
            hi = min(len(idx), t + radius + 1)
            local_max = slab[lo:hi].max(axis=0)
            moved = (1.0 - alpha) * slab[t] + alpha * local_max
            prop[t, event_mask] = np.where(
                local_max[event_mask] > slab[t, event_mask],
                moved[event_mask],
                slab[t, event_mask],
            )
        out[idx] = prop
    return out.astype(np.float32)


def _filemax_scale(pred, power=0.45):
    if power <= 0:
        return pred.copy()
    out = pred.copy()
    for fid in pd.unique(file_ids):
        idx = np.where(file_ids == fid)[0]
        slab = pred[idx]
        scale = np.power(np.clip(slab.max(axis=0, keepdims=True), EPS, 1.0), power)
        out[idx] = slab * scale
    return out.astype(np.float32)


def _adaptive_delta_smooth_final(pred, alpha=0.15):
    if alpha <= 0:
        return pred.copy()
    out = pred.copy()
    for fid in pd.unique(file_ids):
        idx = np.where(file_ids == fid)[0]
        slab = pred[idx]
        if len(idx) <= 1:
            continue
        prop = slab.copy()
        for t in range(len(idx)):
            if t == 0:
                neighbor_avg = (slab[t] + slab[t + 1]) / 2.0
            elif t == len(idx) - 1:
                neighbor_avg = (slab[t - 1] + slab[t]) / 2.0
            else:
                neighbor_avg = (slab[t - 1] + slab[t + 1]) / 2.0
            conf = float(slab[t].max())
            a = alpha * (1.0 - conf)
            prop[t] = (1.0 - a) * slab[t] + a * neighbor_avg
        out[idx] = prop
    return out.astype(np.float32)


def _postproc_member(base_pred, lmax_alpha=0.0, filemax_power=0.0, delta_alpha=0.0):
    x = np.asarray(base_pred, dtype=np.float32).copy()
    x = _local_max_blend(x, alpha=lmax_alpha, radius=1)
    x = _filemax_scale(x, power=filemax_power)
    x = _adaptive_delta_smooth_final(x, alpha=delta_alpha)
    return np.clip(x, 0.0, 1.0).astype(np.float32)


def _find_cv9245_artifacts():
    fold_hits = sorted(Path("/kaggle/input").rglob("moe_p0.60_c0.25_r0.15_post_p0.45_fold1.pt"))
    if not fold_hits:
        raise FileNotFoundError("cv9245 fold weights not found under /kaggle/input")
    weights_dir = fold_hits[0].parent
    script_hits = sorted(weights_dir.rglob("pantanal_infer_only_submission.py"))
    if not script_hits:
        script_hits = sorted(Path("/kaggle/input").rglob("pantanal_infer_only_submission.py"))
    if not script_hits:
        raise FileNotFoundError("pantanal_infer_only_submission.py not found")
    cnn_hits = sorted(Path("/kaggle/input").rglob("student_cnn_2025_plus_2026_nodistill_keepperch*.pt"))
    if not cnn_hits:
        cnn_hits = sorted(Path("/kaggle/input").rglob("student_cnn*.pt"))
    if not cnn_hits:
        raise FileNotFoundError("cv9245 student CNN weight not found")
    return script_hits[0], weights_dir, cnn_hits[0]


def _cv9245_prior_logits_for_files(cv, sites_file, hours_file, tables, n_windows):
    if hasattr(cv, "prior_logits_for_files"):
        return cv.prior_logits_for_files(sites_file, hours_file, tables, n_windows=n_windows)
    sites_flat = np.repeat(sites_file.astype(object), n_windows)
    hours_flat = np.repeat(hours_file.astype(np.int64), n_windows)
    if hasattr(cv, "prior_logits_from_tables"):
        prior_flat = cv.prior_logits_from_tables(sites_flat, hours_flat, tables)
        return prior_flat.reshape(len(sites_file), n_windows, -1)

    p = np.repeat(tables["global_p"][None, :], len(sites_flat), axis=0).astype(np.float32, copy=True)
    for i, (s, h) in enumerate(zip(sites_flat.tolist(), hours_flat.tolist())):
        if str(s) in tables.get("site_to_stats", {}):
            n_s, p_s = tables["site_to_stats"][str(s)]
            w_s = n_s / (n_s + 8.0)
            p[i] = (1.0 - w_s) * p[i] + w_s * p_s
        if int(h) in tables.get("hour_to_stats", {}):
            n_h, p_h = tables["hour_to_stats"][int(h)]
            w_h = n_h / (n_h + 8.0)
            p[i] = (1.0 - w_h) * p[i] + w_h * p_h
    p = np.clip(p, 1e-4, 1.0 - 1e-4)
    return (np.log(p) - np.log1p(-p)).astype(np.float32).reshape(len(sites_file), n_windows, -1)


def run_cv9245_shared_perch():
    start_min = (time.time() - _WALL_START) / 60.0
    if start_min > CV9245_START_CUTOFF_MIN:
        raise TimeoutError(f"skip cv9245: start time {start_min:.1f} min exceeds guard")

    script_path, weights_dir, cnn_weight = _find_cv9245_artifacts()
    spec = importlib.util.spec_from_file_location("cv9245_shared", script_path)
    cv = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(cv)

    print(f"CV9245 sidecar script: {script_path}")
    print(f"CV9245 sidecar weights: {weights_dir}")
    print(f"CV9245 sidecar CNN: {cnn_weight}")

    paths = [Path(p) for p in test_paths]
    n_files = len(paths)
    n_classes = len(PRIMARY_LABELS)
    if n_files == 0:
        raise RuntimeError("no test paths available for cv9245 branch")
    if sc_te.shape[0] != n_files * N_WINDOWS or emb_te.shape[0] != n_files * N_WINDOWS:
        raise RuntimeError(f"unexpected shared Perch shapes: sc_te={sc_te.shape}, emb_te={emb_te.shape}, files={n_files}")

    cv_prior_tables, site_to_idx = cv.build_training_priors(BASE, PRIMARY_LABELS)
    n_sites = max(site_to_idx.values(), default=0) + 1

    student_cnn = cv.StudentCNN(emb_dim=1536, n_classes=n_classes).to("cpu")
    student_cnn.load_state_dict(torch.load(cnn_weight, map_location="cpu"))
    student_cnn.eval()

    mel_tf = cv.torchaudio.transforms.MelSpectrogram(
        sample_rate=cv.SR, n_fft=1024, win_length=1024, hop_length=320,
        n_mels=128, f_min=20, f_max=14000,
    )

    fold_models = []
    for fold in [1, 2, 3, 4]:
        ckpt = weights_dir / f"moe_p0.60_c0.25_r0.15_post_p0.45_fold{fold}.pt"
        if not ckpt.exists():
            raise FileNotFoundError(f"missing cv9245 fold weight: {ckpt}")
        model = cv.ProtoSSM(n_classes=n_classes, d_model=320, n_sites=n_sites).to("cpu")
        model.load_state_dict(torch.load(ckpt, map_location="cpu"))
        model.eval()
        fold_models.append(model)

    t_logits_all = sc_te.astype(np.float32, copy=False).reshape(n_files, N_WINDOWS, n_classes)
    t_emb_all = emb_te.astype(np.float32, copy=False).reshape(n_files, N_WINDOWS, -1)

    all_scores = []
    row_ids_cv = []
    start_ts = time.time()
    batch_files = max(1, int(CV9245_BATCH_FILES))

    for bi in range(0, n_files, batch_files):
        batch_paths = paths[bi:bi + batch_files]
        n_batch = len(batch_paths)
        windows_batch = np.concatenate([cv.read_60s_windows(p) for p in batch_paths], axis=0).astype(np.float32)
        t_logits_batch = t_logits_all[bi:bi + n_batch].copy()
        t_emb_batch = t_emb_all[bi:bi + n_batch]

        with torch.inference_mode():
            w = torch.from_numpy(windows_batch)
            mel = torch.log(mel_tf(w) + 1e-6).unsqueeze(1)
            _, cnn_logits = student_cnn(mel)
            cnn_logits_batch = cnn_logits.numpy().astype(np.float32).reshape(n_batch, N_WINDOWS, n_classes)

        mix_total = 1.0
        t_logits_batch = 0.80 * t_logits_batch + 0.20 * cnn_logits_batch
        t_logits_batch = t_logits_batch / max(mix_total, 1e-6)

        sites_file = np.array([cv.parse_site(p.name) for p in batch_paths], dtype=object)
        hours_file = np.array([cv.parse_hour(p.name) for p in batch_paths], dtype=np.int64)
        prior_batch = _cv9245_prior_logits_for_files(cv, sites_file, hours_file, cv_prior_tables, n_windows=N_WINDOWS)
        t_logits_use_batch = t_logits_batch + 0.45 * prior_batch

        x = torch.from_numpy(t_emb_batch.astype(np.float32, copy=False))
        l = torch.from_numpy(t_logits_use_batch.astype(np.float32, copy=False))
        sid = torch.tensor([site_to_idx.get(str(s), 0) for s in sites_file.tolist()], dtype=torch.long)
        hid = torch.from_numpy(hours_file.astype(np.int64, copy=False))

        with torch.inference_mode():
            probs_sum = torch.zeros((n_batch, N_WINDOWS, n_classes), dtype=torch.float32)
            for model in fold_models:
                probs_sum += torch.sigmoid(model(x, l, sid, hid))
            probs_batch = (probs_sum / float(len(fold_models))).cpu().numpy().astype(np.float32)

        probs_batch = cv.postprocess_probs_filewise(
            probs_batch.reshape(n_batch * N_WINDOWS, n_classes),
            n_windows=N_WINDOWS,
        ).reshape(n_batch, N_WINDOWS, n_classes)

        all_scores.append(probs_batch.reshape(n_batch * N_WINDOWS, n_classes))
        for p in batch_paths:
            stem = p.stem
            for i in range(N_WINDOWS):
                row_ids_cv.append(f"{stem}_{(i + 1) * WINDOW_SEC}")

        done = min(bi + batch_files, n_files)
        if done == n_files or done % 40 == 0:
            elapsed = time.time() - start_ts
            print(f"CV9245 sidecar progress files={done}/{n_files} elapsed={elapsed:.1f}s")

        del windows_batch, t_logits_batch, t_emb_batch, cnn_logits_batch, probs_batch
        gc.collect()

    scores = np.concatenate(all_scores, axis=0).astype(np.float32)
    cv_sub = pd.DataFrame(scores, columns=PRIMARY_LABELS)
    cv_sub.insert(0, "row_id", row_ids_cv)
    if len(sample_sub) == len(cv_sub) and set(sample_sub["row_id"]) == set(cv_sub["row_id"]):
        cv_sub = sample_sub[["row_id"]].merge(cv_sub, on="row_id", how="left")
        cv_sub[PRIMARY_LABELS] = cv_sub[PRIMARY_LABELS].fillna(0.0).astype(np.float32)
    cv_sub.to_csv(CV9245_OUT, index=False)
    print(f"Saved {CV9245_OUT}: {cv_sub.shape}")
    return cv_sub


def _parse_birdnet_label(line):
    text = str(line).strip()
    if not text:
        return ""
    if ";" in text:
        parts = [p.strip() for p in text.split(";")]
        if parts and parts[0].isdigit() and len(parts) > 1:
            text = parts[1]
        else:
            text = parts[0]
    return text.split("_", 1)[0].strip()


def _load_birdnet_labels(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        labels = [_parse_birdnet_label(line) for line in f]
    return [x for x in labels if x]


def _find_birdnet_artifacts():
    roots = [
        Path("/kaggle/input/models/shadiakiki1/birdnet-analyzer/tflite/birdnet_global_6k_v2.4_model_fp32-1/3"),
        Path("/kaggle/input/models/shadiakiki1/birdnet-analyzer"),
        Path("/kaggle/input/birdnet-analyzer"),
    ]
    model_hits = []
    for root in roots:
        if root.exists():
            model_hits.extend(root.rglob("*.tflite"))
    if not model_hits:
        model_hits = [p for p in Path("/kaggle/input").rglob("*.tflite")
                      if "birdnet" in str(p).lower()]
    if not model_hits:
        raise FileNotFoundError("public BirdNET model not found; attach shadiakiki1/birdnet-analyzer to enable it")

    def _model_priority(path):
        s = str(path).lower()
        return (
            0 if "global" in s else 1,
            0 if "6k" in s or "v2.4" in s else 1,
            0 if "fp32" in s else 1,
            len(s),
        )

    model_path = sorted(model_hits, key=_model_priority)[0]
    label_roots = [model_path.parent, model_path.parent.parent, Path("/kaggle/input")]
    label_hits = []
    for root in label_roots:
        if root.exists():
            for pat in ["**/BirdNET_GLOBAL_6K_V2.4_Labels.txt", "**/birdnet*labels*.txt", "**/*birdnet*label*.txt", "**/*.txt"]:
                label_hits.extend(root.rglob(pat))
    best_labels = None
    best_label_path = None
    for label_path in sorted(set(label_hits), key=lambda p: len(str(p))):
        labels = _load_birdnet_labels(label_path)
        if best_labels is None or len(labels) > len(best_labels):
            best_labels = labels
            best_label_path = label_path
    if best_labels is None or len(best_labels) < 1000:
        raise FileNotFoundError(f"BirdNET label file not found near {model_path}")
    return model_path, best_label_path, best_labels


BIRDNET_SR = 48_000
BIRDNET_CHUNK_SEC = 3
BIRDNET_CHUNK_SAMPLES = BIRDNET_SR * BIRDNET_CHUNK_SEC
BIRDNET_N_CHUNKS = 20
BIRDNET_WIN_TO_CHUNKS = []
for w in range(N_WINDOWS):
    ws, we = w * WINDOW_SEC, (w + 1) * WINDOW_SEC
    BIRDNET_WIN_TO_CHUNKS.append([
        j for j in range(BIRDNET_N_CHUNKS)
        if BIRDNET_CHUNK_SEC * j < we and BIRDNET_CHUNK_SEC * (j + 1) > ws
    ])


def run_birdnet_public_branch():
    start_min = _elapsed_min()
    if start_min > BIRDNET_START_CUTOFF_MIN:
        raise TimeoutError(f"skip BirdNET: start time {start_min:.1f} min exceeds guard")

    model_path, label_path, bn_labels = _find_birdnet_artifacts()
    print(f"BirdNET model: {model_path}")
    print(f"BirdNET labels: {label_path} ({len(bn_labels)} labels)")

    label_to_bn = {}
    for i, name in enumerate(bn_labels):
        key = str(name).strip().lower()
        if key and key not in label_to_bn:
            label_to_bn[key] = i

    tax_sci_by_label = taxonomy.set_index("primary_label")["scientific_name"].astype(str).to_dict()
    tax_label_by_sci = {str(v).strip().lower(): k for k, v in tax_sci_by_label.items()}
    bn_idx = np.full(len(cols), -1, dtype=np.int32)
    bn_proxy = {}
    bn_class_mask = np.zeros(len(cols), dtype=bool)

    for ci, label in enumerate(cols):
        sci = tax_sci_by_label.get(label, "").strip().lower()
        idx = label_to_bn.get(sci, -1)
        if idx >= 0:
            bn_idx[ci] = idx
            bn_class_mask[ci] = True
            continue
        if sci and " " in sci:
            genus = sci.split()[0]
            proxy = [i for name, i in label_to_bn.items() if name.startswith(genus + " ")]
            if proxy:
                bn_proxy[ci] = proxy
                bn_class_mask[ci] = True

    if not bn_class_mask.any():
        raise RuntimeError("BirdNET has no scientific-name or genus overlap with submission classes")
    print(f"BirdNET mapped classes: {int(bn_class_mask.sum())}/{len(cols)}")

    try:
        from tflite_runtime.interpreter import Interpreter as _TFLiteInterpreter
        interpreter = _TFLiteInterpreter(model_path=str(model_path), num_threads=4)
    except Exception:
        tf = load_tf()
        interpreter = tf.lite.Interpreter(model_path=str(model_path), num_threads=4)
    interpreter.allocate_tensors()
    in_detail = interpreter.get_input_details()[0]
    out_detail = interpreter.get_output_details()[-1]
    print(f"BirdNET input: {in_detail['shape']} output: {out_detail['shape']}")

    import librosa as _bn_librosa
    from scipy.ndimage import gaussian_filter1d as _bn_gf1d

    paths = [Path(p) for p in test_paths]
    n_files = len(paths)
    if n_files == 0:
        raise RuntimeError("no test paths available for BirdNET branch")
    preds = np.zeros((n_files * N_WINDOWS, len(cols)), dtype=np.float32)
    rows = []
    start_ts = time.time()
    total_samples = 60 * BIRDNET_SR

    for fi, path in enumerate(paths):
        if fi >= 20:
            elapsed = time.time() - start_ts
            projected_total = elapsed / max(fi, 1) * n_files
            projected_end_min = _elapsed_min() + max(0.0, projected_total - elapsed) / 60.0
            if projected_end_min > TOTAL_RUNTIME_GUARD_MIN:
                raise TimeoutError(f"skip BirdNET: projected finish {projected_end_min:.1f} min exceeds guard")

        y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
        if y.ndim == 2:
            y = y.mean(axis=1)
        if sr0 != BIRDNET_SR:
            y = _bn_librosa.resample(y, orig_sr=sr0, target_sr=BIRDNET_SR)
        if len(y) < total_samples:
            y = np.pad(y, (0, total_samples - len(y)))
        else:
            y = y[:total_samples]

        chunks = y.reshape(BIRDNET_N_CHUNKS, BIRDNET_CHUNK_SAMPLES)
        chunk_probs = np.zeros((BIRDNET_N_CHUNKS, len(bn_labels)), dtype=np.float32)
        for j, chunk in enumerate(chunks):
            interpreter.set_tensor(in_detail["index"], chunk[None, :].astype(np.float32))
            interpreter.invoke()
            logits = interpreter.get_tensor(out_detail["index"])[0].astype(np.float32)
            chunk_probs[j] = 1.0 / (1.0 + np.exp(-np.clip(logits, -50, 50)))

        stem = path.stem
        for wi, clist in enumerate(BIRDNET_WIN_TO_CHUNKS):
            wp = chunk_probs[clist].max(axis=0)
            row = fi * N_WINDOWS + wi
            for ci in np.where(bn_idx >= 0)[0]:
                preds[row, ci] = wp[bn_idx[ci]]
            for ci, proxy_idxs in bn_proxy.items():
                preds[row, ci] = max(preds[row, ci], float(wp[proxy_idxs].max()))
            rows.append(f"{stem}_{(wi + 1) * WINDOW_SEC}")

        if fi == 0 or (fi + 1) % 50 == 0 or (fi + 1) == n_files:
            print(f"BirdNET: {fi + 1}/{n_files} elapsed={time.time() - start_ts:.1f}s")

    preds_v = preds.reshape(n_files, N_WINDOWS, len(cols))
    for fi in range(n_files):
        preds_v[fi] = _bn_gf1d(preds_v[fi], sigma=0.65, axis=0, mode="nearest")
    preds = preds_v.reshape(n_files * N_WINDOWS, len(cols))

    bn_sub = pd.DataFrame(np.clip(preds, 0.0, 1.0), columns=cols)
    bn_sub.insert(0, "row_id", rows)
    bn_sub.to_csv(BIRDNET_OUT, index=False)
    print(f"Saved {BIRDNET_OUT}: {bn_sub.shape}")
    return bn_sub, bn_class_mask


print("Executing robust multi-view rank ensemble...")
base_60 = _base_rank_blend(p_proto, p_sed, proto_weight=0.60)
base_55 = _base_rank_blend(p_proto, p_sed, proto_weight=0.55)
pp_v300_like = _postproc_member(base_60, lmax_alpha=0.10, filemax_power=0.45, delta_alpha=0.15)
pp_rank40    = _postproc_member(base_60, lmax_alpha=0.00, filemax_power=0.40, delta_alpha=0.15)

pred = (
    0.50 * base_60 +
    0.25 * base_55 +
    0.15 * pp_v300_like +
    0.10 * pp_rank40
)
pred = np.clip(pred, 0.0, 1.0).astype(np.float32)

branch_registry = []

try:
    cv9245_sub = run_cv9245_shared_perch()
except Exception as e:
    cv9245_sub = None
    print(f"CV9245 sidecar skipped: {e}")

if cv9245_sub is not None:
    branch_registry.append({
        "name": "CV9245",
        "df": cv9245_sub,
        "weight": CV9245_RANK_WEIGHT,
        "class_mask": None,
    })

try:
    birdnet_sub, birdnet_mask = run_birdnet_public_branch()
except Exception as e:
    birdnet_sub, birdnet_mask = None, None
    print(f"BirdNET branch skipped: {e}")

if birdnet_sub is not None:
    branch_registry.append({
        "name": "BirdNET",
        "df": birdnet_sub,
        "weight": BIRDNET_RANK_WEIGHT,
        "class_mask": birdnet_mask,
    })

for branch in branch_registry:
    try:
        branch_arr = _validate_prediction_frame(
            branch["name"], branch["df"], df_proto[["row_id"]], cols)
        pred = _blend_rank_branch(
            pred,
            branch_arr,
            weight=float(branch["weight"]),
            name=branch["name"],
            class_mask=branch.get("class_mask"),
        )
    except Exception as e:
        print(f"{branch['name']} branch validation/blend failed; keeping current ensemble: {e}")

sub = df_proto.copy()
sub[cols] = pred.astype(np.float32)

MIRROR_PAIRS = (
    ("47158son15", "47158son16"),
    ("47158son09", "47158son12"),
    ("47158son02", "47158son14"),
    ("47158son13", "47158son21", "47158son22", "47158son23"),
)
col_to_idx = {l: i for i, l in enumerate(cols)}


def apply_sonotype_mirror(df):
    out = df.copy()
    mirror_count = 0
    for group in MIRROR_PAIRS:
        valid_idx = [col_to_idx[s] for s in group if s in col_to_idx]
        if len(valid_idx) >= 2:
            group_max = out[cols].iloc[:, valid_idx].max(axis=1).to_numpy(np.float32)
            for idx in valid_idx:
                out.iloc[:, idx + 1] = group_max
            mirror_count += len(valid_idx)
    return out, int(mirror_count)


def apply_rare_suppress(df, factor=0.90, margin=0.05):
    out = df.copy()
    rare_count = 0
    try:
        tax_df = pd.read_csv(BASE_PATH / "taxonomy.csv").set_index("primary_label")
        rare_classes = {"Amphibia", "Mammalia", "Reptilia"}
        for ci, species in enumerate(cols):
            if species in tax_df.index and tax_df.loc[species, "class_name"] in rare_classes:
                col_idx = ci + 1
                vals = out.iloc[:, col_idx].to_numpy(np.float32)
                thr = vals.mean() + float(margin)
                out.iloc[:, col_idx] = np.where(vals < thr, vals * float(factor), vals)
                rare_count += 1
    except Exception as e:
        print(f"Rare suppression variant skipped: {e}")
    return out, int(rare_count)


sub, mirror_count = apply_sonotype_mirror(sub)
print(f"Sonotype mirroring applied to {mirror_count} columns.")
sub.to_csv("submission_improved_no_rare.csv", index=False)

rare_variant, rare_count = apply_rare_suppress(sub, factor=0.90, margin=0.05)
rare_variant.to_csv("submission_rare090_variant.csv", index=False)
print(f"Rare-taxon suppression disabled for primary submission; variant saved for {rare_count} classes.")

# Dry-run alignment.
test_paths = list(BASE_PATH.glob("test_soundscapes/*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
if IS_DRY_RUN:
    print("Dry-run detected: Aligning rows with sample_submission.csv")
    sample_public = pd.read_csv(BASE_PATH / "sample_submission.csv")
    template = sub[cols].mean(axis=0).astype(np.float32)
    rare_template = rare_variant[cols].mean(axis=0).astype(np.float32)
    sub = sample_public.copy()
    rare_variant = sample_public.copy()
    for label in cols:
        sub[label] = template[label]
        rare_variant[label] = rare_template[label]
    rare_variant.to_csv("submission_rare090_variant.csv", index=False)

sub[cols] = sub[cols].clip(0.0, 1.0).astype(np.float32)
if list(sub.columns) != ["row_id"] + cols:
    raise RuntimeError("submission columns are not in the expected order")
if sub["row_id"].isna().any() or sub["row_id"].duplicated().any():
    raise RuntimeError("submission row_id contains nulls or duplicates")
if not np.isfinite(sub[cols].to_numpy(np.float32)).all():
    raise RuntimeError("submission contains non-finite predictions")
if ((sub[cols].to_numpy(np.float32) < 0.0).any()
        or (sub[cols].to_numpy(np.float32) > 1.0).any()):
    raise RuntimeError("submission predictions outside [0, 1]")
sub.to_csv(OUT_CSV, index=False)
print(f"Blend and post-processing complete. Saved {OUT_CSV} shape={sub.shape}")
print("Ready for submission!")

# Outputs

| File | Meaning |
|---|---|
| `submission.csv` | Primary Kaggle submission: robust multi-view ProtoSSM/SED blend with optional public sidecars |
| `submission_protossm.csv` | Perch + ProtoSSM + probes + residual branch |
| `submission_sed.csv` | Tucker public distilled SED branch |
| `submission_cv9245_cnnonly_sharedperch.csv` | Optional public CV9245 sidecar branch when available |
| `submission_birdnet_public.csv` | Optional Model_9-style public BirdNET branch when available |
| `submission_improved_no_rare.csv` | Primary ensemble before rare-suppression variant |
| `submission_rare090_variant.csv` | Diagnostic rare-taxon suppression variant; not primary |

Before submission, use `submission.csv`.
